# 3 — Matcher ONNX & TensorRT export

Exports **LightGlue / LighterGlue** matchers to ONNX and compiles FP16 TensorRT engines.

**Prerequisites:**
- `uv sync --group trt` (or `--all-groups`) for `polygraphy`, `tensorrt`
- `pip install onnxsim`
- A CUDA-capable GPU with TensorRT installed

**Prerequisite:** Run [1-export_extractors.ipynb](1-export_extractors.ipynb) and
[2-export_trt_extractors.ipynb](2-export_trt_extractors.ipynb) first.
The extractor `.fp16.engine` files in `weights/euroc/` are used in the visualisation (Section 5).

All export logic lives in `lightglue_dynamo.matcher_export` — no external repos needed.

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    LG = CWD.parent
    REPO_ROOT = LG.parent
elif CWD.name == "LightGlue-ONNX-Jetson":
    LG = CWD
    REPO_ROOT = CWD.parent
else:
    raise SystemExit("Run Jupyter with cwd = LightGlue-ONNX-Jetson or LightGlue-ONNX-Jetson/notebooks")

WEIGHTS = REPO_ROOT / "weights"
OUT_DIR = LG / "weights" / "euroc"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("LG      ", LG)
print("REPO_ROOT", REPO_ROOT)
print("WEIGHTS ", WEIGHTS, WEIGHTS.exists())
print("OUT_DIR ", OUT_DIR, OUT_DIR.exists())

In [ ]:
import sys

if str(LG) not in sys.path:
    sys.path.insert(0, str(LG))

from lightglue_dynamo.matcher_export import export_matcher_onnx, load_lightglue_local
from lightglue_dynamo.extractor_export import build_extractor_trt_engine, simplify_onnx

print("lightglue_dynamo.matcher_export OK")
print("lightglue_dynamo.extractor_export OK")

### Matcher registry

| id | weights file | input_dim (D) | descriptor_dim | n_layers | num_heads | notes |
|----|-------------|---------------|----------------|----------|-----------|-------|
| superpoint | superpoint_lightglue.pth | 256 | 256 | 9 | 4 | old key format (auto-renamed) |
| aliked | aliked_lightglue.pth | 128 | 256 | 9 | 4 | ALIKED-n16 / n32 descriptors |
| raco | raco_aliked_lightglue.pth | 128 | 256 | 9 | 4 | RaCo keypoints + ALIKED descriptors |
| xfeat | xfeat-lighterglue.pt (`matcher.*`) | 64 | 96 | 6 | 1 | LighterGlue, joint checkpoint |

In [ ]:
MATCHER_CONFIGS = [
    dict(
        id="superpoint",
        weights=WEIGHTS / "superpoint_lightglue.pth",
        input_dim=256, desc_dim=256, n_layers=9, num_heads=4,
        prefix=None,
    ),
    dict(
        id="aliked",
        weights=WEIGHTS / "aliked_lightglue.pth",
        input_dim=128, desc_dim=256, n_layers=9, num_heads=4,
        prefix=None,
    ),
    dict(
        id="raco",
        # Architecturally identical to aliked_lightglue: matches RaCo keypoints
        # paired with ALIKED 128-dim descriptors.
        weights=WEIGHTS / "raco_aliked_lightglue.pth",
        input_dim=128, desc_dim=256, n_layers=9, num_heads=4,
        prefix=None,
    ),
    dict(
        id="xfeat",
        # Joint checkpoint: extractor weights under "extractor.*",
        # LighterGlue matcher weights under "matcher.*".
        weights=WEIGHTS / "xfeat-lighterglue.pt",
        input_dim=64, desc_dim=96, n_layers=6, num_heads=1,
        prefix="matcher.",
    ),
]

K = 256  # fixed keypoint budget — must match the extractor export

print(f"{'id':12} {'input_dim':>9} {'desc_dim':>8} {'n_layers':>8} {'num_heads':>9}  weights")
for cfg in MATCHER_CONFIGS:
    status = "OK" if cfg["weights"].is_file() else "MISSING"
    print(f"{cfg['id']:12} {cfg['input_dim']:>9} {cfg['desc_dim']:>8} "
          f"{cfg['n_layers']:>8} {cfg['num_heads']:>9}  {cfg['weights'].name}  [{status}]")

### ONNX export

All four matchers share the same I/O contract:

```
Inputs:
  kpts0  (1, K, 2)   isotropic-normalised keypoints for image 0
  kpts1  (1, K, 2)   isotropic-normalised keypoints for image 1
  desc0  (1, K, D)   descriptors for image 0  (D = input_dim)
  desc1  (1, K, D)   descriptors for image 1

Outputs:
  matches0  (M, 2)   match pairs [idx_in_kpts0, idx_in_kpts1]
  mscores0  (M,)     match confidence scores
```

K is **fixed at 256**; M (number of matches) is dynamic.

Internally, `LightGlueExporter` concatenates the pair into `(2, K, *)` for the dynamo
model's batched-pair format, then strips the batch-index column from the match output.

In [ ]:
onnx_paths: dict[str, Path] = {}

for cfg in MATCHER_CONFIGS:
    cid = cfg["id"]
    if not cfg["weights"].is_file():
        print(f"[SKIP] {cid}: weights not found at {cfg['weights']}")
        continue

    out = OUT_DIR / f"{cid}_lightglue_kp{K}.onnx"
    print(f"Exporting {cid} matcher → {out.name} ...")

    export_matcher_onnx(
        cfg["weights"],
        out,
        num_keypoints=K,
        input_dim=cfg["input_dim"],
        descriptor_dim=cfg["desc_dim"],
        num_heads=cfg["num_heads"],
        n_layers=cfg["n_layers"],
        state_dict_prefix=cfg["prefix"],
        opset=17,
        device="cpu",
    )

    print(f"[OK]   {out.name}  ({out.stat().st_size / 1e6:.1f} MB)")
    onnx_paths[cid] = out

print(f"\nExported {len(onnx_paths)}/{len(MATCHER_CONFIGS)} matcher ONNX models.")

### ONNX simplification (onnxsim)

Runs constant folding and dead-node elimination on each matcher ONNX.
Output: `*_sim.onnx` alongside the originals.

In [ ]:
sim_paths: dict[str, Path] = {}

for cid, onnx_path in onnx_paths.items():
    sim_path = onnx_path.with_name(onnx_path.stem + "_sim.onnx")
    try:
        simplify_onnx(onnx_path, sim_path)
        ratio = sim_path.stat().st_size / onnx_path.stat().st_size
        print(f"[OK]   {onnx_path.name}")
        print(f"       -> {sim_path.name}  ({ratio:.1%} of original)")
        sim_paths[cid] = sim_path
    except Exception as exc:
        print(f"[ERR]  {onnx_path.name}: {exc} — using original")
        sim_paths[cid] = onnx_path

print(f"\nSimplified {sum(1 for p in sim_paths.values() if '_sim' in p.stem)}/{len(onnx_paths)} models.")

### TensorRT FP16 engine build

Builds one FP16 engine per simplified ONNX. Engines are saved as `*_sim.fp16.engine`.

In [ ]:
engine_paths: dict[str, Path] = {}

for cid, sim_path in sim_paths.items():
    engine_path = sim_path.with_suffix(".fp16.engine")
    try:
        build_extractor_trt_engine(sim_path, engine_path, fp16=True)
        print(f"[OK]   {engine_path.name}  ({engine_path.stat().st_size / 1e6:.1f} MB)")
        engine_paths[cid] = engine_path
    except Exception as exc:
        print(f"[FAIL] {cid}: {exc}")

print(f"\nBuilt {len(engine_paths)}/{len(sim_paths)} matcher TRT engines.")

### Test — ORT CUDA vs TRT

Runs both ORT CUDA (on the simplified ONNX) and TRT (on the engine) with the same
random inputs. Reports match counts, max absolute score difference, and TRT timing.

> Random inputs produce very few or zero mutual-nearest-neighbour matches — this
> test checks numerical consistency between ORT and TRT, not match quality.

In [ ]:
import time

import numpy as np
import onnxruntime as ort
from polygraphy.backend.common import BytesFromPath
from polygraphy.backend.trt import EngineFromBytes, TrtRunner

PROVIDERS_ORT = ["CUDAExecutionProvider", "CPUExecutionProvider"]
N_WARMUP = 5
N_TIMED  = 20

rng = np.random.default_rng(42)  # fixed seed so ORT and TRT see identical inputs

for cid, engine_path in engine_paths.items():
    sim_path = sim_paths[cid]
    cfg = next(c for c in MATCHER_CONFIGS if c["id"] == cid)
    D = cfg["input_dim"]

    kpts0 = rng.random((1, K, 2), dtype=np.float32) * 2 - 1
    kpts1 = rng.random((1, K, 2), dtype=np.float32) * 2 - 1
    desc0 = rng.standard_normal((1, K, D)).astype(np.float32)
    desc1 = rng.standard_normal((1, K, D)).astype(np.float32)
    feed  = {"kpts0": kpts0, "kpts1": kpts1, "desc0": desc0, "desc1": desc1}

    # ORT reference
    ort_matches, ort_scores = None, None
    try:
        sess = ort.InferenceSession(str(sim_path), providers=PROVIDERS_ORT)
        ort_matches, ort_scores = sess.run(None, feed)
    except Exception as exc:
        print(f"[ORT SKIP] {cid}: {exc}")

    # TRT warmup + timed
    with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
        for _ in range(N_WARMUP):
            runner.infer(feed_dict=feed)
        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            trt_out = runner.infer(feed_dict=feed)
        elapsed_ms = (time.perf_counter() - t0) / N_TIMED * 1000

    trt_matches = trt_out["matches0"]
    trt_scores  = trt_out["mscores0"]

    print(f"\n=== {cid}  TRT {elapsed_ms:.2f} ms/pair ===")
    print(f"  TRT  matches={trt_matches.shape}  scores={trt_scores.shape}")

    if ort_matches is not None:
        print(f"  ORT  matches={ort_matches.shape}")
        if ort_matches.shape == trt_matches.shape and ort_matches.shape[0] > 0:
            ort_idx = np.lexsort(ort_matches.T[::-1])
            trt_idx = np.lexsort(trt_matches.T[::-1])
            diff = float(np.abs(ort_scores[ort_idx] - trt_scores[trt_idx]).max())
            print(f"  max_score_diff={diff:.6f}")

### Match visualisation

Chains each extractor engine with its corresponding matcher engine and visualises
the matches on image pairs.
Timing is reported per stage: **extract**, **describe** (RaCo only), and **match**.


In [ ]:
import re
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from polygraphy.backend.common import BytesFromPath
from polygraphy.backend.trt import EngineFromBytes, TrtRunner

from lightglue_dynamo import viz
from lightglue_dynamo.extractor_export import EXTRACTOR_REGISTRY

# --- paths (only cell 1 needed) ---
K = 256
asset_dir = LG / "assets"

N_WARMUP = 5
N_TIMED  = 20


def load_imgs_bgr(h: int, w: int) -> list[np.ndarray]:
    imgs_bgr = []
    # for fname in ["sacre_coeur1.jpg", "sacre_coeur2.jpg"]:
    # for fname in ["DSC_0410.JPG", "DSC_0411.JPG"]:
    for fname in ["debug1.png", "debug2.png"]:
        p = asset_dir / fname
        if p.is_file():
            imgs_bgr.append(cv2.resize(cv2.imread(str(p)), (w, h)))
        else:
            print(f"[WARN] {fname} not found — using random noise")
            imgs_bgr.append(np.random.randint(0, 255, (h, w, 3), dtype=np.uint8))
    return imgs_bgr


def normalize_isotropic(kpts: np.ndarray, h: int, w: int) -> np.ndarray:
    shift = np.array([w, h], dtype=np.float32) / 2
    scale = max(w, h) / 2
    return ((kpts.astype(np.float32) - shift) / scale).astype(np.float32)


def run_extractor(engine_path, spec, images_bgr) -> tuple[dict, float]:
    frames = []
    for img in images_bgr:
        f = spec.preprocess(img)
        while f.ndim > 3:
            f = f[0]
        frames.append(f)
    x = np.stack(frames, axis=0).astype(np.float32)
    feed = {spec.input_name: x}
    with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
        for _ in range(N_WARMUP):
            runner.infer(feed_dict=feed)
        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            out = runner.infer(feed_dict=feed)
        return out, (time.perf_counter() - t0) / N_TIMED


def run_describe(engine_path, feed_dict) -> tuple[dict, float]:
    with TrtRunner(EngineFromBytes(BytesFromPath(str(engine_path)))) as runner:
        for _ in range(N_WARMUP):
            runner.infer(feed_dict=feed_dict)
        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            out = runner.infer(feed_dict=feed_dict)
        return out, (time.perf_counter() - t0) / N_TIMED


def kpts_to_pixels(kpts_b: np.ndarray, anisotropic: bool, h: int, w: int) -> np.ndarray:
    if anisotropic:
        size = np.array([w, h], dtype=np.float32)
        return (kpts_b.astype(np.float32) + 1.0) * size / 2.0
    return kpts_b.astype(np.float32)


# --- ALIKED describe helper (used only for the RaCo pipeline) ---
# The exported ALIKED TRT engine fuses SDDH sampling at its own detected keypoints
# and exposes no input port for external coordinates, so we use PyTorch directly.
_aliked_model_cache = None


def _get_aliked() -> torch.nn.Module:
    global _aliked_model_cache
    if _aliked_model_cache is None:
        from lightglue_dynamo.models.aliked.aliked import ALIKED as _ALIKED
        _aliked_model_cache = _ALIKED(
            model_name="aliked-n16",
            device="cpu",
            pretrained_path=str(WEIGHTS / "aliked-n16.pth"),
        ).eval()
    return _aliked_model_cache


def aliked_describe_at(kpts_px: np.ndarray, img_bgr: np.ndarray) -> np.ndarray:
    """ALIKED-SDDH descriptors at given pixel (x,y) keypoints."""
    aliked = _get_aliked()
    h, w = img_bgr.shape[:2]
    img_t = torch.from_numpy(
        img_bgr[..., ::-1].astype(np.float32) / 255.0
    ).permute(2, 0, 1).unsqueeze(0)
    wh = torch.tensor([[w - 1, h - 1]], dtype=torch.float32)
    kpts_n = 2.0 * torch.from_numpy(kpts_px.astype(np.float32)) / wh - 1.0
    with torch.no_grad():
        feature_map, _ = aliked.extract_dense_map(img_t)
        descs_list, _ = aliked.desc_head(feature_map, [kpts_n])
    return descs_list[0].cpu().numpy()


def _stem_to_eid(stem: str):
    stem = re.sub(r"_b\d+_h\d+_w\d+_kp\d+(_sim)?(\.fp16\.engine)?$", "", stem)
    for eid in sorted(EXTRACTOR_REGISTRY, key=len, reverse=True):
        if stem.startswith(eid):
            return eid
    return None


# ---------------------------------------------------------------------------
# Pipeline registry
# (label, extractor_engine, matcher_engine, describe_engine, anisotropic, raco_describe, H, W)
# All engine filenames are relative to OUT_DIR.
# ---------------------------------------------------------------------------
PIPELINES = [
    ("SuperPoint + LightGlue",
     f"superpoint_b2_h480_w752_kp{K}_sim.fp16.engine",
     f"superpoint_lightglue_kp{K}_sim.fp16.engine",
     None, False, False, 480, 752),
    ("ALIKED-n16 + LightGlue",
     f"aliked_n16_b2_h480_w768_kp{K}_sim.fp16.engine",
     f"aliked_lightglue_kp{K}_sim.fp16.engine",
     None, True, False, 480, 768),
    ("RaCo + ALIKED-describe + LightGlue",
     f"raco_b2_h480_w768_kp{K}_sim.fp16.engine",
     f"raco_lightglue_kp{K}_sim.fp16.engine",
     f"aliked_n16_describe_b2_h480_w768_kp{K}_sim.fp16.engine",
     False, True, 480, 768),
    ("xFeat + LighterGlue",
     f"xfeat_b2_h480_w752_kp{K}_sim.fp16.engine",
     f"xfeat_lightglue_kp{K}_sim.fp16.engine",
     None, False, False, 480, 752),
]


for label, ext_stem, mat_stem, desc_stem, aniso, raco_describe, H, W in PIPELINES:
    imgs_bgr = load_imgs_bgr(H, W)
    ext_path   = OUT_DIR / ext_stem
    match_path = OUT_DIR / mat_stem

    if not ext_path.is_file():
        print(f"[SKIP] {label}: extractor engine not found ({ext_stem})")
        continue
    if not match_path.is_file():
        print(f"[SKIP] {label}: matcher engine not found ({mat_stem})")
        continue

    t_describe = 0.0

    if raco_describe:
        raco_spec = EXTRACTOR_REGISTRY["raco"]
        ext_out, t_extract = run_extractor(ext_path, raco_spec, imgs_bgr)

        kpts_px = ext_out["keypoints"].astype(np.float32)   # (2, K, 2) pixel-space
        n0 = n1 = kpts_px.shape[1]

        wh_norm = np.array([W - 1, H - 1], dtype=np.float32)
        kpts_n_b = 2.0 * kpts_px / wh_norm - 1.0  # (2, K, 2) anisotropic
        imgs_rgb_nchw = np.stack(
            [img[..., ::-1].astype(np.float32) / 255.0 for img in imgs_bgr], axis=0
        ).transpose(0, 3, 1, 2)  # (2, 3, H, W)
        desc_feed = {"image": imgs_rgb_nchw, "kpts_n": kpts_n_b}
        desc_path = (OUT_DIR / desc_stem) if desc_stem else None

        if desc_path is not None and desc_path.is_file():
            desc_out, t_describe = run_describe(desc_path, desc_feed)
            descs_b = desc_out["descriptors"]  # (2, K, 128)
        else:
            print(f"  [WARN] {label}: describe engine missing — PyTorch fallback")
            t0 = time.perf_counter()
            descs0 = aliked_describe_at(kpts_px[0], imgs_bgr[0])
            descs1 = aliked_describe_at(kpts_px[1], imgs_bgr[1])
            t_describe = time.perf_counter() - t0
            descs_b = np.stack([descs0, descs1], axis=0)
    else:
        eid = _stem_to_eid(ext_stem.replace("_sim.fp16.engine", ""))
        if eid is None:
            print(f"[SKIP] {label}: cannot resolve extractor id from {ext_stem!r}")
            continue
        spec    = EXTRACTOR_REGISTRY[eid]
        ext_out, t_extract = run_extractor(ext_path, spec, imgs_bgr)

        kpts_b   = ext_out["keypoints"]
        descs_b  = ext_out["descriptors"]
        scores_b = ext_out["keypoint_scores"]
        num_b    = ext_out.get("num_keypoints")
        n0 = int(num_b[0]) if num_b is not None else int((scores_b[0] > 0).sum())
        n1 = int(num_b[1]) if num_b is not None else int((scores_b[1] > 0).sum())
        kpts_px  = kpts_to_pixels(kpts_b, aniso, H, W)

    # --- Matcher (TRT) ---
    kpts0_n = normalize_isotropic(kpts_px[0], H, W)[None]
    kpts1_n = normalize_isotropic(kpts_px[1], H, W)[None]
    desc0   = descs_b[0:1].astype(np.float32)
    desc1   = descs_b[1:2].astype(np.float32)

    feed_mat = {"kpts0": kpts0_n, "kpts1": kpts1_n, "desc0": desc0, "desc1": desc1}
    with TrtRunner(EngineFromBytes(BytesFromPath(str(match_path)))) as runner:
        for _ in range(N_WARMUP):
            runner.infer(feed_dict=feed_mat)
        t0 = time.perf_counter()
        for _ in range(N_TIMED):
            m_out = runner.infer(feed_dict=feed_mat)
        t_match = (time.perf_counter() - t0) / N_TIMED

    matches0 = m_out["matches0"]
    mscores0 = m_out["mscores0"]

    valid    = (matches0[:, 0] < n0) & (matches0[:, 1] < n1)
    matches0 = matches0[valid]
    mscores0 = mscores0[valid]

    mkpts0 = kpts_px[0][matches0[:, 0]]
    mkpts1 = kpts_px[1][matches0[:, 1]]

    # --- Timing summary ---
    if t_describe > 0:
        timing_str = (
            f"extract {t_extract*1e3:.1f} ms"
            f"  | describe {t_describe*1e3:.1f} ms"
            f"  | match {t_match*1e3:.1f} ms"
            f"  | total {(t_extract + t_describe + t_match)*1e3:.1f} ms"
        )
    else:
        timing_str = (
            f"extract {t_extract*1e3:.1f} ms"
            f"  | match {t_match*1e3:.1f} ms"
            f"  | total {(t_extract + t_match)*1e3:.1f} ms"
        )
    print(f"{label}: {len(matches0)} matches  |  {timing_str}")

    # --- Visualise ---
    viz.plot_images(imgs_bgr, titles=[f"image1 ({n0} kpts1)",
                                      f"image2 ({n1} kpts)"], dpi=96)
    if len(matches0) > 0:
        colors = plt.cm.plasma(
            (mscores0 - mscores0.min()) / max(mscores0.max() - mscores0.min(), 1e-6)
        ).tolist()
        viz.plot_matches(mkpts0, mkpts1, color=colors, lw=0.8, a=0.7, ps=5)
    viz.add_text(0, f"{label}\n{len(matches0)} matches", fs=10)
    viz.add_text(1, timing_str, fs=8)
    plt.show()


## Three-stage ALIKED pipeline parity vs single-engine + LightGlue

Compares the legacy `aliked_n16` single-engine pipeline (backbone + score_head + DKD + desc_head
all in one TRT engine) against the new three-stage pipeline:

1. `aliked_n16_dense_b{1,2}` — dense headless extractor (backbone + score_head only).
2. Python DKD on the engine's `score_map` (production version is a custom CUDA kernel).
3. `aliked_n16_dhead_b{1,2}` — vectorised SDDH desc-head-only (no backbone re-run).

Both pipelines feed `aliked_lightglue_kp256` for matching. This cell reports cross-image and
self-match agreement on `assets/sacre_coeur{1,2}.jpg`. Build the new engines via notebook 2's
"Three-stage pipeline" section first.


In [ ]:
import cv2
import numpy as np
import torch
from polygraphy.backend.common import BytesFromPath
from polygraphy.backend.trt import EngineFromBytes, TrtRunner

from lightglue_dynamo.preprocessors.aliked import ALIKEDPreprocessor
from lightglue_dynamo.models.aliked.soft_detect import DKD

H, W = 480, 768
MAX_KP = 256

LEGACY  = OUT_DIR / f"aliked_n16_b2_h{H}_w{W}_kp{MAX_KP}_sim.fp16.engine"
DENSE   = OUT_DIR / f"aliked_n16_dense_b2_h{H}_w{W}_sim.fp16.engine"
DHEAD   = OUT_DIR / f"aliked_n16_dhead_b2_h{H}_w{W}_kp{MAX_KP}_sim.fp16.engine"
MATCHER = OUT_DIR / f"aliked_lightglue_kp{MAX_KP}_sim.fp16.engine"

for p, lab in [(LEGACY, "legacy aliked_n16 engine"),
               (DENSE,  "dense headless extractor (notebook 2)"),
               (DHEAD,  "desc-head-only engine (notebook 2)"),
               (MATCHER, "ALIKED-LightGlue matcher")]:
    assert p.is_file(), f"missing {lab}: {p}"

pre = ALIKEDPreprocessor()
imgs = [LG / "assets" / "sacre_coeur1.jpg", LG / "assets" / "sacre_coeur2.jpg"]
batch = []
for p in imgs:
    bgr = cv2.resize(cv2.imread(str(p)), (W, H))
    proc = pre.preprocess(bgr).astype(np.float32)
    while proc.ndim > 3:
        proc = proc[0]
    batch.append(proc)
x_b2 = np.stack(batch, axis=0)

# Legacy single-engine
with TrtRunner(EngineFromBytes(BytesFromPath(str(LEGACY)))) as r:
    leg = r.infer(feed_dict={"images": x_b2})
kpts_leg = leg["keypoints"].astype(np.float32)
desc_leg = leg["descriptors"].astype(np.float32)

# New: dense -> python DKD -> dhead
with TrtRunner(EngineFromBytes(BytesFromPath(str(DENSE)))) as r:
    dense_out = r.infer(feed_dict={"image": x_b2})
score_map_np   = dense_out["score_map"]
feature_map_np = dense_out["feature_map"]

dkd = DKD(radius=2, top_k=MAX_KP, scores_th=0.0, n_limit=MAX_KP).eval()
with torch.no_grad():
    kp_n_list, _, _ = dkd(torch.from_numpy(score_map_np.astype(np.float32)))
kpts_new = np.zeros((2, MAX_KP, 2), dtype=np.float32)
for b in range(2):
    n = min(len(kp_n_list[b]), MAX_KP)
    kpts_new[b, :n] = kp_n_list[b].numpy()[:n]

with TrtRunner(EngineFromBytes(BytesFromPath(str(DHEAD)))) as r:
    dh = r.infer(feed_dict={"feature_map": feature_map_np, "kpts_n": kpts_new})
desc_new = dh["descriptors"].astype(np.float32)

def match(kp0, kp1, d0, d1):
    with TrtRunner(EngineFromBytes(BytesFromPath(str(MATCHER)))) as r:
        return r.infer(feed_dict={"kpts0": kp0, "kpts1": kp1, "desc0": d0, "desc1": d1})

m_leg = match(kpts_leg[0:1], kpts_leg[1:2], desc_leg[0:1], desc_leg[1:2])
m_new = match(kpts_new[0:1], kpts_new[1:2], desc_new[0:1], desc_new[1:2])
print(f"LEGACY: {len(m_leg['matches0'])} matches  mean mscore={m_leg['mscores0'].mean():.4f}")
print(f"NEW   : {len(m_new['matches0'])} matches  mean mscore={m_new['mscores0'].mean():.4f}")

def to_px(kp_n, h, w):
    return ((kp_n + 1) * 0.5 * np.array([w-1, h-1], dtype=np.float32)).astype(np.float32)

leg_kp0_px = to_px(kpts_leg[0], H, W); leg_kp1_px = to_px(kpts_leg[1], H, W)
new_kp0_px = to_px(kpts_new[0], H, W); new_kp1_px = to_px(kpts_new[1], H, W)
leg_pairs = [(leg_kp0_px[int(i)], leg_kp1_px[int(j)]) for i, j in m_leg["matches0"]]
new_pairs = [(new_kp0_px[int(i)], new_kp1_px[int(j)]) for i, j in m_new["matches0"]]

def reproduce(src, tgt, tol=1.5):
    n = 0
    for sp0, sp1 in src:
        for tp0, tp1 in tgt:
            if np.linalg.norm(sp0 - tp0) < tol and np.linalg.norm(sp1 - tp1) < tol:
                n += 1; break
    return n

if leg_pairs and new_pairs:
    a = reproduce(leg_pairs, new_pairs); b = reproduce(new_pairs, leg_pairs)
    print(f"  legacy matches reproduced by new (<1.5 px): {a}/{len(leg_pairs)} ({a/len(leg_pairs)*100:.1f}%)")
    print(f"  new    matches present in legacy (<1.5 px): {b}/{len(new_pairs)} ({b/len(new_pairs)*100:.1f}%)")

# Self-match sanity
x_self = np.stack([batch[0], batch[0]], axis=0)
with TrtRunner(EngineFromBytes(BytesFromPath(str(LEGACY)))) as r:
    leg_self = r.infer(feed_dict={"images": x_self})
m_self_leg = match(leg_self["keypoints"][0:1].astype(np.float32),
                   leg_self["keypoints"][1:2].astype(np.float32),
                   leg_self["descriptors"][0:1].astype(np.float32),
                   leg_self["descriptors"][1:2].astype(np.float32))
print(f"\nself-match LEGACY: {len(m_self_leg['matches0'])} matches  mean mscore={m_self_leg['mscores0'].mean():.4f}")

with TrtRunner(EngineFromBytes(BytesFromPath(str(DENSE)))) as r:
    dense_self = r.infer(feed_dict={"image": x_self})
with torch.no_grad():
    kp_n_self, _, _ = dkd(torch.from_numpy(dense_self["score_map"].astype(np.float32)))
kpts_self = np.zeros((2, MAX_KP, 2), dtype=np.float32)
for b in range(2):
    n = min(len(kp_n_self[b]), MAX_KP); kpts_self[b, :n] = kp_n_self[b].numpy()[:n]
with TrtRunner(EngineFromBytes(BytesFromPath(str(DHEAD)))) as r:
    desc_self = r.infer(feed_dict={"feature_map": dense_self["feature_map"], "kpts_n": kpts_self})["descriptors"].astype(np.float32)
m_self_new = match(kpts_self[0:1], kpts_self[1:2], desc_self[0:1], desc_self[1:2])
print(f"self-match NEW   : {len(m_self_new['matches0'])} matches  mean mscore={m_self_new['mscores0'].mean():.4f}")


### Match visualisation: legacy vs new pipeline (sample asset pair)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 11), dpi=96)
fig.suptitle("LightGlue matches: legacy aliked_n16 vs new dense+dhead pipeline", fontsize=12)

img_a = cv2.cvtColor(cv2.resize(cv2.imread(str(imgs[0])), (W, H)), cv2.COLOR_BGR2RGB)
img_b = cv2.cvtColor(cv2.resize(cv2.imread(str(imgs[1])), (W, H)), cv2.COLOR_BGR2RGB)

def draw_pair(ax, kp0_px, kp1_px, matches, mscores, title):
    canvas = np.concatenate([img_a, img_b], axis=1)
    ax.imshow(canvas)
    ax.set_title(f"{title}  ({len(matches)} matches, mean mscore={mscores.mean():.3f})", fontsize=10)
    ax.axis("off")
    if len(matches) == 0:
        return
    cmap = plt.get_cmap("viridis")
    for (i, j), s in zip(matches, mscores):
        p0 = kp0_px[int(i)]
        p1 = kp1_px[int(j)] + np.array([W, 0], dtype=np.float32)
        c = cmap(float(np.clip(s, 0.0, 1.0)))
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=c, linewidth=0.7, alpha=0.8)
        ax.scatter([p0[0], p1[0]], [p0[1], p1[1]], c=[c], s=5)

draw_pair(axes[0], leg_kp0_px, leg_kp1_px, m_leg["matches0"], m_leg["mscores0"],
          "Legacy aliked_n16_b2 -> LightGlue")
draw_pair(axes[1], new_kp0_px, new_kp1_px, m_new["matches0"], m_new["mscores0"],
          "Dense (Engine A) + python DKD + dhead (Engine B) -> LightGlue")

plt.tight_layout()
plt.show()
